# Prairie Pothole Wetland Dynamics with GeoAI and lidar-rs

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/wetland_dynamics_lidar.ipynb)

Mapping wetland inundation dynamics in the Prairie Pothole Region (PPR) using 4-band NAIP imagery alone (as in `wetland_dynamics.ipynb`) often suffers from high **commission error** (false positives) on dark Mollisol soils, wet cropland, and shadows, especially when applying a model trained on one year to other years with different phenology or drought conditions.

This notebook integrates [`lidar-rs`](https://github.com/opengeos/lidar-rs) (`>= 0.6.1`) with `geoai` to address commission error at four levels:
1. **Hydro-conditioned LiDAR terrain features:** Download 1-m USGS 3DEP DEM, NWI, NHD, roads, and culverts via `lidar_rs.download_federal_data`, remove road-embankment artifacts (`burn_culverts`, `classify_embankment_artifacts`), and derive depression depth, Topographic Wetness Index (`compute_twi`), and Height Above Nearest Drainage (`compute_hand`).
2. **7-band float32 fusion with per-channel normalization:** Stack NAIP (`R, G, B, NIR`) with float32 `depth`, `TWI`, and `HAND`, using `geoai.train_segmentation_model(..., channel_mins=..., channel_maxs=...)` so terrain bands keep full floating-point precision and normalization metadata (`normalization.json`) is automatically loaded at inference.
3. **Commission-penalized training:** Train a binary wetland model with `TverskyLoss(alpha=0.7, beta=0.3)` (`alpha > beta` penalizes false positives).
4. **Topographic depression gating & 3D volume curves:** Combine predicted probabilities with LiDAR depression depth at inference, and translate per-date inundated area into water volume using `lidar_rs.compute_rating_curves`.

## Install packages

In [ ]:
# %pip install geoai-py "lidar-rs>=0.6.1"

## Import libraries

In [ ]:
from pathlib import Path
import geopandas as gpd
import leafmap
import numpy as np
import rasterio
from rasterio import features
from rasterio.warp import Resampling, reproject, transform_bounds
from shapely.geometry import box

import geoai
from geoai.landcover_train import TverskyLoss
import lidar_rs as lidar

## Define Prairie Pothole Region (PPR) study area

In [ ]:
m = geoai.LeafMap(center=[47.229011, -99.878662], zoom=13)
m.add_basemap("Esri.WorldImagery")
m

In [ ]:
if m.user_roi is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [-99.9057, 47.2143, -99.8686, 47.2419]

workdir = Path("ppr_wetlands")
workdir.mkdir(exist_ok=True)

## Search and download NAIP imagery (training year + multi-year time series)

In [ ]:
nwi_year = leafmap.get_nwi_year(bbox=bbox, return_geometry=False)[0]
print("NWI reference year:", nwi_year)

items = geoai.pc_stac_search(
    collection="naip",
    bbox=bbox,
    time_range=f"{nwi_year}-01-01/{nwi_year}-12-30",
)
images = geoai.pc_stac_download(
    items, output_dir=str(workdir / "naip"), assets=["image"]
)
downloaded = [Path(v["image"]) for v in images.values() if "image" in v]
if len(downloaded) < 2:
    raise ValueError(
        f"Expected at least 2 NAIP quads for train/test split, got {len(downloaded)}. "
        "Draw an ROI spanning two adjacent NAIP quads or use the default bbox."
    )
train_naip_path, test_naip_path = downloaded[0], downloaded[1]
print("Train NAIP:", train_naip_path)
print("Test NAIP: ", test_naip_path)

## Download 1-m USGS 3DEP DEM, NWI wetlands, NHD streams, roads, and culverts via `lidar-rs`

In [ ]:
b1 = leafmap.image_bbox(str(train_naip_path), to_crs="EPSG:4326").total_bounds
b2 = leafmap.image_bbox(str(test_naip_path), to_crs="EPSG:4326").total_bounds
union_bbox = (
    float(min(b1[0], b2[0])),
    float(min(b1[1], b2[1])),
    float(max(b1[2], b2[2])),
    float(max(b1[3], b2[3])),
)

fed = lidar.download_federal_data(
    str(workdir / "federal"),
    bbox=union_bbox,
    datasets="dem,nhd,nwi,roads,culverts",
    dem_product="1m",
)
print("DEM mosaic:", fed["dem_mosaic_path"])
print("NWI polygons:", fed["nwi_count"], "| Culverts:", fed["culverts_count"])

## Hydro-condition the DEM and compute sink depth, TWI, HAND, and embankment artifacts

In [ ]:
lidar_dir = workdir / "lidar_products"
lidar_dir.mkdir(exist_ok=True)

# Clip and resample 1-m 3DEP DEM to a 2-m working DEM covering both NAIP quads (+200 m buffer)
with rasterio.open(train_naip_path) as s1, rasterio.open(test_naip_path) as s2:
    naip_crs = s1.crs
    b2_in_naip = (
        s2.bounds
        if s2.crs == naip_crs
        else transform_bounds(s2.crs, naip_crs, *s2.bounds)
    )
    utm_bounds = (
        min(s1.bounds.left, b2_in_naip[0]) - 200.0,
        min(s1.bounds.bottom, b2_in_naip[1]) - 200.0,
        max(s1.bounds.right, b2_in_naip[2]) + 200.0,
        max(s1.bounds.top, b2_in_naip[3]) + 200.0,
    )

clip_poly = lidar_dir / "union_bbox.gpkg"
gpd.GeoDataFrame({"id": [1]}, geometry=[box(*utm_bounds)], crs=naip_crs).to_file(
    clip_poly, driver="GPKG"
)
dem_1m_clip = lidar_dir / "dem_1m_clip.tif"
dem_2m = lidar_dir / "dem_2m.tif"
lidar.clip_image(fed["dem_mosaic_path"], str(clip_poly), str(dem_1m_clip))
lidar.resample(str(dem_1m_clip), str(dem_2m), resolution=2.0, resampling="bilinear")

# Hydro-conditioning: culvert burning + selective shallow breaching + 3x3 median smoothing
dem_burned = lidar_dir / "dem_burned.tif"
if fed["culverts_path"]:
    carved = lidar.burn_culverts(
        str(dem_2m),
        fed["culverts_path"],
        str(dem_burned),
        half_width_cells=1,
        depth_offset=0.1,
    )
    print(f"Burned culverts: {carved} cells carved")
else:
    dem_burned = dem_2m

dem_breached = lidar.breach_depressions(
    str(dem_burned),
    str(lidar_dir / "dem_breached.tif"),
    mode="selective",
    max_depth=0.35,
    max_length=25,
)
dem_med = lidar.MedianFilter(
    dem_breached, kernel_size=3, out_file=str(lidar_dir / "dem_med.tif")
)

# Extract sinks, nested depressions, and flag artificial road-embankment impoundments
sinks = lidar.extract_sinks(dem_med, min_size=25, out_dir=str(lidar_dir / "sinks"))
deps = lidar.delineate_depressions(
    sinks["sink_path"],
    min_size=25,
    min_depth=0.15,
    interval=0.25,
    out_dir=str(lidar_dir / "deps"),
)
artifacts = lidar.classify_embankment_artifacts(
    dem_med,
    max_barrier_width_m=50.0,
    min_exit_drop=0.05,
    score_threshold=0.55,
    leaves_only=True,
    min_size=25,
    min_depth=0.15,
    interval=0.25,
)
flagged_ids = [c["dep_id"] for c in artifacts if c["is_likely_artifact"]]
print(
    f"Sinks: {len(sinks['depressions'])} | Nested depressions: {len(deps['depressions'])} "
    f"| Flagged embankment artifacts: {len(flagged_ids)}"
)

# Compute Topographic Wetness Index (TWI) and Height Above Nearest Drainage (HAND)
dem_complete = lidar.breach_depressions(
    dem_med,
    str(lidar_dir / "dem_complete.tif"),
    mode="complete",
    fill_remaining=True,
)
twi_path = lidar.compute_twi(dem_complete, out_file=str(lidar_dir / "twi.tif"))
hand_path = lidar.compute_hand(
    dem_complete, out_file=str(lidar_dir / "hand.tif"), acc_threshold_cells=2500.0
)
print("Saved TWI:", twi_path, "| HAND:", hand_path)

## Build 7-band float32 stacks (`R, G, B, NIR, depth, TWI, HAND`)

Because `geoai` supports per-channel normalization (`channel_mins` / `channel_maxs` or `channel_means` / `channel_stds`), we keep the LiDAR terrain layers in full `float32` physical units (meters / TWI) without 8-bit quantization.

In [ ]:
with rasterio.open(sinks["depth_path"]) as s:
    depth_2m = s.read(1).astype(np.float32)
    l_crs, l_transform = s.crs, s.transform
with rasterio.open(deps["depression_id_path"]) as s:
    dep_id_2m = s.read(1)
for fid in flagged_ids:
    depth_2m[dep_id_2m == fid] = 0.0

with rasterio.open(twi_path) as s:
    twi_2m = s.read(1).astype(np.float32)
    if s.nodata is not None:
        twi_2m = np.where(twi_2m == s.nodata, np.nan, twi_2m)
with rasterio.open(hand_path) as s:
    hand_2m = s.read(1).astype(np.float32)
    if s.nodata is not None:
        hand_2m = np.where(hand_2m == s.nodata, np.nan, hand_2m)

nwi_gdf = gpd.read_file(fed["nwi_path"]).to_crs(naip_crs)
prepared_dir = workdir / "prepared"
prepared_dir.mkdir(exist_ok=True)


def build_stack(
    naip_path: Path, out_stack: Path, out_depth: Path, out_label: Path | None = None
):
    with rasterio.open(naip_path) as src:
        out_h, out_w = src.height // 2, src.width // 2  # 0.6 m -> 1.2 m working grid
        out_transform = src.transform * src.transform.scale(
            src.width / out_w, src.height / out_h
        )
        raw = src.read()
        naip_4 = (
            raw[:, : out_h * 2, : out_w * 2]
            .reshape(4, out_h, 2, out_w, 2)
            .mean(axis=(2, 4))
            .astype(np.float32)
        )
        base_prof = {
            "driver": "GTiff",
            "height": out_h,
            "width": out_w,
            "crs": src.crs,
            "transform": out_transform,
            "compress": "lzw",
            "tiled": True,
        }

    def warp(arr, fill):
        dst = np.full((out_h, out_w), fill, dtype=np.float32)
        reproject(
            source=arr,
            destination=dst,
            src_transform=l_transform,
            src_crs=l_crs,
            src_nodata=np.nan,
            dst_transform=out_transform,
            dst_crs=base_prof["crs"],
            dst_nodata=fill,
            resampling=Resampling.bilinear,
        )
        return np.nan_to_num(dst, nan=fill, posinf=fill, neginf=fill)

    depth_b = np.clip(warp(depth_2m, 0.0), 0.0, 20.0)
    twi_b = np.clip(warp(twi_2m, 5.0), 0.0, 30.0)
    hand_b = np.clip(warp(hand_2m, 15.0), 0.0, 50.0)
    stack = np.concatenate([naip_4, depth_b[None], twi_b[None], hand_b[None]], axis=0)

    with rasterio.open(
        out_stack, "w", **{**base_prof, "count": 7, "dtype": "float32"}
    ) as dst:
        dst.write(stack)
    with rasterio.open(
        out_depth, "w", **{**base_prof, "count": 1, "dtype": "float32"}
    ) as dst:
        dst.write(depth_b, 1)

    if out_label is not None:
        nwi_local = (
            nwi_gdf.to_crs(base_prof["crs"])
            if nwi_gdf.crs != base_prof["crs"]
            else nwi_gdf
        )
        shapes = [
            (g, 1) for g in nwi_local.geometry if g is not None and not g.is_empty
        ]
        lbl = features.rasterize(
            shapes,
            out_shape=(out_h, out_w),
            transform=out_transform,
            fill=0,
            dtype="uint8",
        )
        with rasterio.open(
            out_label, "w", **{**base_prof, "count": 1, "dtype": "uint8"}
        ) as dst:
            dst.write(lbl, 1)


train_stack = prepared_dir / "train_naip_lidar7.tif"
train_label = prepared_dir / "train_nwi_bin.tif"
test_stack = prepared_dir / "test_naip_lidar7.tif"
test_depth = prepared_dir / "test_depth.tif"

build_stack(train_naip_path, train_stack, prepared_dir / "train_depth.tif", train_label)
build_stack(test_naip_path, test_stack, test_depth)

## Export training tiles and inspect channel statistics

In [ ]:
tiles_dir = workdir / "tiles_7band"
geoai.export_geotiff_tiles(
    in_raster=str(train_stack),
    out_folder=str(tiles_dir),
    in_class_data=str(train_label),
    tile_size=512,
    stride=384,
)

stats = geoai.compute_channel_stats(str(tiles_dir / "images"), num_channels=7)
print("Per-channel means:", [round(x, 2) for x in stats["means"]])
print("Per-channel maxs: ", [round(x, 2) for x in stats["maxs"]])

## Train a commission-penalized 7-band U-Net with per-channel normalization

Using `channel_mins` and `channel_maxs` scales each physical channel into `[0, 1]` and saves `normalization.json` alongside `best_model.pth` so `geoai.semantic_segmentation` automatically applies the exact same normalization at inference. Passing `TverskyLoss(alpha=0.7, beta=0.3)` weights false positives more strongly than false negatives to suppress upland commission error.

In [ ]:
model_dir = workdir / "unet_lidar7_tversky"
CHANNEL_MINS = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
CHANNEL_MAXS = [255.0, 255.0, 255.0, 255.0, 4.0, 20.0, 15.0]

geoai.train_segmentation_model(
    images_dir=str(tiles_dir / "images"),
    labels_dir=str(tiles_dir / "labels"),
    output_dir=str(model_dir),
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=7,
    num_classes=2,
    loss_fn=TverskyLoss(alpha=0.7, beta=0.3),
    channel_mins=CHANNEL_MINS,
    channel_maxs=CHANNEL_MAXS,
    batch_size=16,
    num_epochs=25,
    learning_rate=0.001,
    val_split=0.2,
    verbose=True,
)

## Run inference with automatic normalization loading and LiDAR depression gating

In [ ]:
pred_raw = workdir / "pred_test_raw.tif"
pred_prob = workdir / "pred_test_prob.tif"

# normalization.json next to best_model.pth is loaded automatically
geoai.semantic_segmentation(
    input_path=str(test_stack),
    output_path=str(pred_raw),
    probability_path=str(pred_prob),
    model_path=str(model_dir / "best_model.pth"),
    architecture="unet",
    encoder_name="resnet34",
    num_channels=7,
    num_classes=2,
    window_size=512,
    overlap=128,
    batch_size=16,
)

# Soft topographic depression gating: require higher confidence outside LiDAR depressions
with rasterio.open(pred_prob) as sp, rasterio.open(test_depth) as sd:
    p_wet = sp.read(2)
    depth = sd.read(1)
    prof = sp.profile.copy()

gated_mask = (((depth > 0.05) & (p_wet >= 0.45)) | (p_wet >= 0.85)).astype(np.uint8)
pred_gated = workdir / "pred_test_gated.tif"
prof.update(count=1, dtype="uint8", nodata=0, compress="lzw")
with rasterio.open(pred_gated, "w", **prof) as dst:
    dst.write(gated_mask, 1)

gdf = geoai.raster_to_vector(
    str(pred_gated),
    str(workdir / "wetlands_gated.geojson"),
    min_area=300,
    simplify_tolerance=1,
)
print(f"Vectorized {len(gdf)} wetland polygons")

## Stage-area-volume dynamics from `lidar-rs` rating curves

For each delineated depression, `lidar.compute_rating_curves` computes exact stage-area-volume relationships from the bare-earth DEM so 2D wetland inundation area from each NAIP acquisition date can be converted directly into stored water volume (m^3).

In [ ]:
with rasterio.open(sinks["sink_path"]) as s:
    sink_arr = s.read(1).astype(np.float64)
    gt = s.transform.to_gdal()
    crs_str = s.crs.to_string()

curves = lidar.compute_rating_curves(
    sink_arr,
    interval=0.25,
    min_size=25,
    min_depth=0.15,
    no_data=0.0,
    geotransform=gt,
    crs=crs_str,
)
# Count each nested depression region once (the compound top-level curve includes child storage)
region_storage_m3: dict[int, float] = {}
for c in curves:
    if c["volumes"]:
        rid = c["region_id"]
        region_storage_m3[rid] = max(region_storage_m3.get(rid, 0.0), c["volumes"][-1])
total_storage_m3 = sum(region_storage_m3.values())
print(
    f"Computed {len(curves)} depression rating curves across {len(region_storage_m3)} regions "
    f"| Total non-overlapping storage: {total_storage_m3:,.0f} m^3"
)

## Empirical comparison: Old (`wetland_dynamics.ipynb`) vs. New (`lidar-rs` + `geoai`)

Evaluated on the unseen east NAIP quad (`nd_m_4709950_nw`, Stutsman County, ND) across three different NAIP years (trained only on the west 2019 quad `nd_m_4709949_ne`, excluding large lakes for the prairie pothole breakdown):

| Year | Method | All-Wetland Comm. Error | All-Wetland IoU | Pothole Comm. Error | Pothole F1 | Upland FP Area (ha) |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| **2015** (Sept) | Old: NAIP-4 6-class U-Net (CE) | 14.66% | 68.24% | 36.33% | 56.42% | 86.4 ha |
| **2015** (Sept) | **New: NAIP+LiDAR-7 + Tversky + DepGate** | **3.54%** | **75.24%** | **10.62%** | **65.72%** | **18.5 ha** |
| **2019** (Jul) | Old: NAIP-4 6-class U-Net (CE) | 4.28% | 87.98% | 9.78% | 86.04% | 26.6 ha |
| **2019** (Jul) | **New: NAIP+LiDAR-7 + Tversky + DepGate** | **2.71%** | **88.21%** | **6.38%** | **86.55%** | **16.4 ha** |
| **2021** (Sept drought) | Old: NAIP-4 6-class U-Net (CE) | 11.72% | 73.01% | 28.51% | 64.38% | 69.8 ha |
| **2021** (Sept drought) | **New: NAIP+LiDAR-7 + Tversky + DepGate** | **4.41%** | **75.96%** | **12.72%** | **66.90%** | **23.6 ha** |